# Ok-k-mer

## Imports

In [1]:
import os
from random import randint
import numpy as np
import pandas as pd
from collections import Counter
from ete3 import Tree
from itertools import combinations, product
from scipy.stats import binom, geom, spearmanr, pearsonr

pd.options.display.max_rows = 40

In [2]:
# import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mplpatches
%matplotlib inline

plt.rcParams['figure.dpi'] = 300
plt.rcParams['agg.path.chunksize'] = 10000

In [3]:
# import plotnine

from plotnine import \
    ggplot, aes, theme, themes, element_blank, element_line, element_text, mapping, \
    geom_point, geom_jitter, geom_line, geom_bar, geom_boxplot, geom_violin, geom_histogram, \
    ggtitle, xlab, ylab, geom_vline, geom_hline, geom_abline, geom_col, geom_tile, \
    scale_x_discrete, scale_y_discrete, scale_x_continuous, scale_y_continuous, \
    scale_color_discrete, scale_fill_manual, scale_fill_hue, scale_fill_gradient, scale_color_manual, scale_color_hue, \
    guides, after_stat, \
    xlim, ylim
from plotnine.options import figure_size
from mizani.formatters import percent_format

# plotnine theme
def theme_tufte_func(base_size=19, base_family="sans", rotate=True, figure_size=None):
    ''' Plotnine plotting theme. '''
    thm = themes.theme_bw(base_size=base_size, base_family = base_family) + \
        theme(
            legend_background = element_blank(),
            legend_key = element_blank(),
            panel_background = element_blank(),
            panel_border = element_blank(),
            strip_background = element_blank(),
            plot_background = element_blank(),
            panel_grid = element_blank(),
            axis_line = element_line(colour = "black", size = 1),
            axis_text_y = element_text(colour = "black")
        )

    if rotate:
        thm += theme(axis_text_x=element_text(rotation=45, hjust=1))
    if figure_size is not None:
        thm += theme(figure_size=figure_size)
    return thm


## Helper functions

In [ ]:
/scratch4/blangme2/sjhwang/ok_k_mer/data/cyanobacteriales/o__Cyanobacteriales.nwk
/scratch4/blangme2/sjhwang/ok_k_mer/data/chlamydiales/o__Chlamydiales.nwk

In [ ]:
def dist_tree_calc(filename1):
    ''' Calculate tree distance. '''
    tree1= Tree(filename1,format=1)
    # print('Tree from', filename1, 'len:', len(tree1))
    dist_tree={}
    samples = [i.name for i in tree1.get_leaves()]
    num_sample = len(samples)
    for i in range(num_sample):
        tax_i = samples[i]
        for j in range(i+1,num_sample):
            tax_j = samples[j]
            dist_tree[(min(tax_i, tax_j)+".fa", max(tax_i, tax_j)+".fa")] = tree1.get_distance(tax_i, tax_j, topology_only=False)
    # print('num samples:', num_sample)
    # print('xxx:', num_sample * (num_sample-1)/2)
    # print('tree len:', len(dist_tree))
    return dist_tree



In [ ]:
tree_d_path = '/Users/stephenhwang/Documents/projects/langmead_lab/MEM-ANI/data/cyano/cyanobacteriales_tree_distances.sorted.txt'

names = set()
for x in yield_file_lines(tree_d_path):
    g1, g2, d = x.split('\t')
    names.add(g1)
    names.add(g2)

tree_dict = {_:{_:0 for _ in names} for _ in names}

for x in yield_file_lines(tree_d_path):
    g1, g2, d = x.split('\t')
    d = float(d)
    tree_dict[g1][g2] = d
    tree_dict[g2][g1] = d

len(tree_dict)

NameError: name 'yield_file_lines' is not defined

In [ ]:
def calc_corr(l1, l2, pearson=False, prnt=False):
    if pearson:
        corr, pval = pearsonr(l1, l2)
    else:
        corr, pval = spearmanr(l1, l2)
    if prnt:
        print("Correlation coefficient:", corr)
        print("p-value:", pval)
    return corr, pval

In [27]:
# parse data into dfs

dir_path = '/scratch4/blangme2/sjhwang/ok_k_mer/data/cyanobacteriales/dashingfull_genomes_ncbi'
dir_path = '/scratch4/blangme2/sjhwang/ok_k_mer/data/chlamydiales/dashingfull_genomes_ncbi'

# Get list of files in directory that start with 'distance_'
files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f)) and f.startswith('distance_')]

# Create a dictionary to store dataframes
dashing_dfs = {}

# Read each file into a dataframe, skipping first line
for file in files:
    file_path = os.path.join(dir_path, file)
    try:
        # Skip first line and read tab-separated data
        df = pd.read_csv(file_path, sep='\t', header=None, skiprows=1)
        df.index = df.iloc[:, 0].str.replace('genomes_ncbi/', '')
        df = df.iloc[:, 1:]
        df.columns = df.index
        dashing_dfs[file] = df
        print(f"Successfully read {file}")
    except Exception as e:
        print(f"Error reading {file}: {str(e)}")

# Print summary of loaded dataframes
print(f"\nLoaded {len(dashing_dfs)} files")
for file, df in dashing_dfs.items():
    print(f"{file}: {df.shape}")

Successfully read distance_k10.txt
Successfully read distance_k11.txt
Successfully read distance_k12.txt
Successfully read distance_k13.txt
Successfully read distance_k14.txt
Successfully read distance_k15.txt
Successfully read distance_k16.txt
Successfully read distance_k17.txt
Successfully read distance_k18.txt
Successfully read distance_k19.txt
Successfully read distance_k20.txt
Successfully read distance_k21.txt
Successfully read distance_k22.txt
Successfully read distance_k23.txt
Successfully read distance_k24.txt
Successfully read distance_k25.txt
Successfully read distance_k26.txt
Successfully read distance_k27.txt
Successfully read distance_k28.txt
Successfully read distance_k29.txt
Successfully read distance_k30.txt
Successfully read distance_k31.txt
Successfully read distance_k32.txt
Error reading distance_k33.txt: No columns to parse from file
Successfully read distance_k5.txt
Successfully read distance_k6.txt
Successfully read distance_k7.txt
Successfully read distance_k8.t

In [ ]:
# Filter for just the distance_k files and sort them by k value
distance_files = sorted([f for f in dashing_dfs.keys() if f.startswith('distance_k')],
                       key=lambda x: int(x.replace('distance_k', '').replace('.txt', '')))

# Create list of pairs and their distances
pairs = []
k_values = {}

# Process first file to get genome names and create pairs
first_df = dashing_dfs[distance_files[0]]
genomes = first_df.iloc[0, 1:].tolist()  # Get genome names from first row

# Clean genome names by removing 'genomes_ncbi/'
genomes = [g.replace('genomes_ncbi/', '') for g in genomes]

# Create all pairs of genomes (upper triangle only)
for i in range(len(genomes)):
    for j in range(i+1, len(genomes)):
        # Sort lexicographically and join with '_'
        pair = '_'.join(sorted([genomes[i], genomes[j]]))
        pairs.append(pair)

# Initialize dataframe with pairs as index
merged_df = pd.DataFrame(index=pairs)

# Add data from each distance file
for file in distance_files:
    k = int(file.replace('distance_k', '').replace('.txt', ''))
    df = dashing_dfs[file]

    # Extract distances for each pair
    distances = []
    for i in range(len(genomes)):
        for j in range(i+1, len(genomes)):
            distances.append(df.iloc[i+1, j+1])

    # Add as new column
    merged_df[f'k_{k}'] = distances

# Clean up the index name
merged_df.index.name = 'genome_pair'


# Add metadat columns to merged_df
genome_pairs = []
genomes_1s = []
genomes_2s = []
tree_distances = []

for index in merged_df.index:
    # Split the genome pair
    genome1 = "_".join(index.split("_", 2)[:2])
    genome2 = "_".join(index.split("_", 2)[2:])

    # Get distance from tree_dict
    distance = tree_dict[genome1][genome2]

    genome_pairs.append(index)
    genomes_1s.append(genome1)
    genomes_2s.append(genome2)
    tree_distances.append(distance)

# Add tree_distance column to merged_df
merged_df['genome_pair'] = genome_pairs
merged_df['genome_1'] = genomes_1s
merged_df['genome_2'] = genomes_2s
merged_df['tree_distance'] = tree_distances

print(f"Final dataframe shape: {merged_df.shape}")
merged_df
